# Act 4: Planetary Descent & Soft Landing

Landing an autonomous probe requires continuous attitude control and rocket throttle regulation under unknown gravity and turbulent crosswinds. Because simulated physics engines are non-differentiable, we use Evolution Strategies (ES) to optimize a neural flight policy. ES perturbs candidate policy weights, evaluates their landing performance in parallel environments, and updates the network towards higher rewards. Fine-tuning population size, mutation noise, and learning rate steers the lander towards stable, fuel-efficient touchdowns.

In [ ]:
# Setup and package installation
REPO_URL = "https://github.com/placeholder/meridian"
# Option 1: Install from GitHub in Google Colab
# !pip install -q git+$REPO_URL

# Option 2: Install from local editable package
# !pip install -e /path/to/meridian-workshop

import sys
from pathlib import Path
repo_dir = Path.cwd()
while repo_dir.parent != repo_dir and not (repo_dir / "meridian").exists():
    repo_dir = repo_dir.parent
if (repo_dir / "meridian").exists() and str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))


In [ ]:
import os
TEAM_ID = os.environ.get("MERIDIAN_TEAM_ID", "your-team-name")

from meridian.state import load_state
from meridian.ui import briefing, dashboard, debrief
state = load_state(TEAM_ID)


In [ ]:
briefing(4)

### Physical Flight Envelope

The physical conditions of this landing reflect your mission history across earlier acts. Surface gravity and lateral wind were fixed by the exoplanet discovered in Act 1. Available thruster fuel and sensor noise reflect your anomaly handling during the Act 2 cruise.

In [ ]:
from meridian.act4_lander import build_conditions, train, evaluate, apply_to_state, plot_learning_curve, human_vs_ai
from meridian import seeding
conditions = build_conditions(state)
print(f'Descent Conditions: Gravity={conditions["gravity"]:.2f} m/s², Wind={conditions["wind"]:.2f} m/s², Fuel={conditions["fuel_pct"]:.1f}%, Noise=σ {conditions["sensor_noise"]:.3f}')

### Human vs AI Warm-up

How does an unguided free-fall descent compare against an active flight control algorithm? We benchmark an unguided baseline against our hand-coded heuristic proportional-derivative (PD) controller. Notice how active feedback regulation stabilizes attitude and slows vertical touchdown velocity.

In [ ]:
# Compare heuristic PD controller against unguided free-fall
human_vs_ai(conditions)

### YOUR TURN: Train Neural Flight Policy with Evolution Strategies

Adjust your Evolution Strategies optimization hyperparameters below. A larger population explores more candidate weight perturbations per generation but requires more CPU steps. Setting warm_start=True initializes policy weights from our pre-trained baseline checkpoint.

In [ ]:
# YOUR TURN: Adjust Act 4 Evolution Strategies hyperparameters
knobs = {
    "iterations": 15,
    "pop_size": 20,
    "sigma": 0.08,
    "lr": 0.04,
    "warm_start": True,
}

In [ ]:
# Train neural flight controller via Evolution Strategies
policy, history = train(conditions, **knobs)
eval_seed = seeding.seed_for(TEAM_ID, 'act4_eval')
eval_res = evaluate(policy, conditions, n_episodes=100, seed=eval_seed)
apply_to_state(state, eval_res)
print(f'Act 4 Score: {state.act4.score:.1f}/400 | Landing Rate: {eval_res["landing_rate"] * 100.0:.1f}% | Mean Reward: {eval_res["mean_reward"]:.1f}')

In [ ]:
# Plot ES training learning curve
import matplotlib.pyplot as plt
plot_learning_curve(history)
plt.show()

In [ ]:
debrief(state, 4)

In [ ]:
state.save()
dashboard(state)